# Anonymization Audit: Can the Face Be Reconstructed?

Loads the saved anonymized OBJ from notebook 48 and tries to get the face back.
Goal: demonstrate that the saved file contains no face data and that naive
reconstruction (hole-filling) does not recover the subject's features.

**Inputs**
- `SubjectN.obj` - original photogrammetry scan (face present)
- `SubjectN_anon.obj` - output of notebook 48 (face region deleted, no texture)


In [1]:
import os

import numpy as np
import pyvista as pv
import trimesh

import cedalion
import cedalion.io
import cedalion.plots

pv.set_jupyter_backend('server')

SUBJECT_NUMBER = 17
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'

orig_path = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
anon_path = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}_anon.obj'
anon_dir = os.path.dirname(anon_path)
stem = f'Subject{SUBJECT_NUMBER}_anon'
siblings = sorted(f for f in os.listdir(anon_dir) if f.startswith(stem))
print('Files written by the anonymization save step:')
for f in siblings:
    print(f'  {f}')
assert f'{stem}.obj' in siblings, 'expected .obj not found -- run notebook 48 first'
assert f'{stem}.mtl' not in siblings, 'unexpected .mtl on disk -- texture leaked'
assert f'{stem}.jpg' not in siblings, 'unexpected .jpg on disk -- face pixels leaked'
print('\nOK: only the .obj was written. No .mtl / .jpg on disk.')


Files written by the anonymization save step:
  Subject17_anon.obj

OK: only the .obj was written. No .mtl / .jpg on disk.


## 1. Load original and anonymized scans

In [2]:
surface_orig = cedalion.io.read_einstar_obj(orig_path)
surface_anon = cedalion.io.read_einstar_obj(anon_path)

print(f'Original:    {surface_orig.nvertices:>8,} verts, {surface_orig.nfaces:>8,} faces')
print(f'Anonymized:  {surface_anon.nvertices:>8,} verts, {surface_anon.nfaces:>8,} faces')
print(f'Removed:     {surface_orig.nvertices - surface_anon.nvertices:>8,} verts, '
      f'{surface_orig.nfaces - surface_anon.nfaces:>8,} faces')

anon_visual = surface_anon.mesh.visual
print(f'\nAnonymized visual type: {type(anon_visual).__name__}')
has_uv = getattr(anon_visual, 'uv', None) is not None
has_img = getattr(anon_visual, 'image', None) is not None
print(f'  has UV map:      {has_uv}')
print(f'  has texture img: {has_img}')


Original:    1,284,667 verts, 2,223,716 faces
Anonymized:   475,536 verts,  948,028 faces
Removed:      809,131 verts, 1,275,688 faces

Anonymized visual type: ColorVisuals
  has UV map:      False
  has texture img: False


## 2. Side-by-side render

Left: original subject. Right: reloaded anonymized scan. The anonymized side
should show a visible hole where the face was.

In [3]:
pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
pvplt.add_text('Original', font_size=10)
cedalion.plots.plot_surface(pvplt, surface_orig, opacity=1.0)

pvplt.subplot(0, 1)
pvplt.add_text('Anonymized (reloaded)', font_size=10)
cedalion.plots.plot_surface(pvplt, surface_anon, opacity=1.0)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:45309/index.html?ui=P_0x7f67384fc090_0&reconnect=auto" class="pyvi…

## Conclusion

- Only the `.obj` geometry leaves the process. No `.mtl`, no `.jpg`, no texture
  raster with the subject's face pixels.
- The reloaded anonymized mesh has a visible hole where the face region was.
- The saved file contains only geometry for the non-face regions; the face
  vertices and the entire texture image are gone, so the subject's face
  cannot be reconstructed from it.